# AMEX Enterprise Credit Risk Platform
## Notebook 04 — Feature Engineering
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Data Preparation**. Sprint 1, Notebook 4 of 18. Depends on Notebooks 01-03 (reads `project_config.json`, `notebook_02_summary.json`, and `Data_Validation/data_validation_report.json`) -- run those first if you have not already.

**What this notebook does:** extends Notebook 02's aggregated feature stores (mean/std/min/max/last per raw column) with three new families of engineered features:
- **Trend/delta features** -- a genuine per-customer linear trend (least-squares slope) and a first-to-last delta, computed across each customer's real statement history. This requires a second, dedicated streaming pass over the raw `train_data.csv` / `test_data.csv` -- the trend across all 13 months cannot be recovered from Notebook 02's already-aggregated output, only from the raw statement sequence. That second pass is a deliberate cost, not an oversight.
- **Ratio features** -- last-vs-typical, range, and coefficient-of-variation, derived cheaply from Notebook 02's existing stats (no raw re-scan needed for these).
- **Interaction terms** -- pairwise products among the features Notebook 03 already found most correlated with the target, live -- not an arbitrary or memorized selection.

Feature **selection** is handled as ranking, not deletion: every new engineered feature gets a live-computed correlation-with-target, saved as a ranked report. This notebook does not delete any column on its own judgment -- Notebook 05 (Model Development) makes the final feature-pruning call using proper model-based importance, which is a stronger criterion than a single correlation number.

**Zero-fabrication rule (same as Notebooks 01-03):** every number, feature, and file below is computed live by this cell during this run.

**Run the single code cell below, once.** Idempotent -- every output file is written to a fixed path and overwritten in place on every re-run. This cell streams the two raw files a second time (in addition to Notebook 02's pass), so expect meaningful wall-clock time.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01-03
# =============================================================================
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-03")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB03_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_03_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB03_SUMMARY_PATH, "run 03_data_validation_eda.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix} -- this notebook reads its outputs.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB03_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB03_SUMMARY = json.load(f)

DATA_ROOT = Path(PROJECT_CONFIG["data_root"])
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
WARP_THREAD_COUNT = (
    PROJECT_CONFIG.get("resource_limits", {}).get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)  # 95% of logical cores when Notebook 01's resource_limits block is present;
   # falls back to the raw core count on an older project_config.json.

TRAIN_CSV = DATA_ROOT / "train_data.csv"
TEST_CSV = DATA_ROOT / "test_data.csv"
TRAIN_FULL_PATH = Path(NB02_SUMMARY["output_files"]["train_full_features.parquet"])
TEST_FEATURES_PATH = Path(NB02_SUMMARY["output_files"]["test_features.parquet"])
TRAIN_SPLIT_PATH = Path(NB02_SUMMARY["output_files"]["train_split.csv"])
TEST_SPLIT_PATH = Path(NB02_SUMMARY["output_files"]["test_split.csv"])

VALIDATION_REPORT_PATH = PILLAR_DIRS["data_validation"] / "data_validation_report.json"
if not VALIDATION_REPORT_PATH.exists():
    raise FileNotFoundError(f"{VALIDATION_REPORT_PATH} not found.\nFix: run 03_data_validation_eda.ipynb first.")
with open(VALIDATION_REPORT_PATH, "r", encoding="utf-8") as f:
    VALIDATION_REPORT = json.load(f)

for _p in (TRAIN_CSV, TEST_CSV, TRAIN_FULL_PATH, TEST_FEATURES_PATH, TRAIN_SPLIT_PATH, TEST_SPLIT_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}")

print(f"Loaded config from    : {CONFIG_PATH}")
print(f"Loaded NB02 summary   : {NB02_SUMMARY_PATH}")
print(f"Loaded NB03 summary   : {NB03_SUMMARY_PATH}")
print(f"Loaded validation report: {VALIDATION_REPORT_PATH}")
print(f"train_data.csv        : {TRAIN_CSV.stat().st_size / 1e9:.2f} GB")
print(f"test_data.csv         : {TEST_CSV.stat().st_size / 1e9:.2f} GB")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

try:
    import polars as pl
except ImportError:
    raise ImportError("Missing required package: polars\nFix: pip install polars")

logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (WARP 6.4, Concurrency)")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD BASE FEATURE STORES (FROM NOTEBOOK 02) & LIVE SCHEMA DETECTION
# =============================================================================
_section("SECTION 3: Load Base Feature Stores & Live Schema Detection")

_t0 = time.time()
train_base = pl.read_parquet(str(TRAIN_FULL_PATH))
test_base = pl.read_parquet(str(TEST_FEATURES_PATH))
print(f"Loaded train_full_features.parquet: {train_base.shape[0]:,} x {train_base.shape[1]} "
      f"({time.time() - _t0:.1f}s)")
print(f"Loaded test_features.parquet      : {test_base.shape[0]:,} x {test_base.shape[1]}")

CATEGORICAL = ["B_30", "B_38", "D_63", "D_64", "D_66", "D_68",
               "D_114", "D_116", "D_117", "D_120", "D_126"]
with open(TRAIN_CSV, "r", encoding="utf-8") as f:
    raw_header = f.readline().strip().split(",")
raw_feature_cols = [c for c in raw_header if c not in ("customer_ID", "S_2")]
numeric_raw_cols = [c for c in raw_feature_cols if c not in CATEGORICAL]
print(f"Numeric raw columns eligible for trend/delta features: {len(numeric_raw_cols)}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: TREND & DELTA FEATURES -- DEDICATED RAW-DATA STREAMING PASS
# =============================================================================
_section("SECTION 4: Trend & Delta Features (Raw-Data Streaming Pass)")


def build_trend_features(csv_path: Path, numeric_cols: list) -> "pl.DataFrame":
    """Second streaming pass over the raw file: for every numeric column,
    computes a per-customer least-squares trend (slope of value vs. statement
    sequence number) and a first-to-last delta.

    WARP alignment: same streaming/schema-override/explicit-sort discipline
    as Notebook 02's build_customer_feature_store(). Slope is computed via
    the covariance/variance identity (cov(t, y) / var(t)) rather than a
    per-row Python loop -- this stays fully vectorized inside Polars'
    streaming engine instead of falling back to a slow, non-WARP-aligned
    per-customer Python computation.

    Correctness note: pl.cov(t, y) automatically excludes any row where y is
    null (pairwise-complete), but a plain pl.col("t").var() does NOT -- it
    would use every statement's time index regardless of which ones paired
    with a missing y, silently mismatching cov's denominator against a
    different set of points and giving a systematically wrong slope whenever
    a customer has any missing value in that column (verified against an
    independent numpy computation before shipping this notebook). The fix is
    to mask t to null everywhere y is null, per column, before taking its
    variance -- so var(t) is computed over exactly the same points cov(t, y)
    used.

    Correctness note (raw-value inf cleaning, before cov/var): same root
    cause as Notebook 02's build_customer_feature_store() -- the real AMEX
    raw files contain literal "inf"/"-inf" tokens in some numeric columns,
    which parse to genuine floating-point infinity once explicitly typed as
    Float32 here. cov()/var() are both variance-identity computations, so an
    uncleaned inf value produces NaN in _cov_{c}/_var_t_{c} (not a Polars
    null), which the pl.when(...).is_not_null() guard below does NOT catch --
    NaN passes is_not_null(). Cleaning inf -> null on the raw numeric columns
    immediately after scanning, before _t_idx/cov/var ever see them, removes
    this failure mode at its origin, the same way Notebook 02 does.
    """
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in numeric_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in numeric_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
    )

    agg_exprs = []
    for c in numeric_cols:
        agg_exprs += [
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None)
              .var().alias(f"_var_t_{c}"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.col(c).last().alias(f"_last_{c}"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)

    final_exprs = ["customer_ID"]
    with_col_exprs = []
    for c in numeric_cols:
        with_col_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}"))
            .otherwise(None)
            .alias(f"{c}_trend_slope")
        )
        with_col_exprs.append(
            (pl.col(f"_last_{c}") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta")
        )

    result = grouped.with_columns(with_col_exprs).select(
        ["customer_ID"] + [f"{c}_trend_slope" for c in numeric_cols] + [f"{c}_trend_delta" for c in numeric_cols]
    ).sort("customer_ID")

    return result.collect(engine="streaming")


print(f"Streaming {TRAIN_CSV.name} for trend/delta features ({TRAIN_CSV.stat().st_size / 1e9:.2f} GB)...")
_t0 = time.time()
train_trend = build_trend_features(TRAIN_CSV, numeric_raw_cols)
_train_trend_seconds = time.time() - _t0
print(f"Computed trend/delta for {train_trend.shape[0]:,} train customers "
      f"({(train_trend.shape[1] - 1)} new columns) in {_train_trend_seconds:.1f}s")

print(f"\nStreaming {TEST_CSV.name} for trend/delta features ({TEST_CSV.stat().st_size / 1e9:.2f} GB)...")
_t0 = time.time()
test_trend = build_trend_features(TEST_CSV, numeric_raw_cols)
_test_trend_seconds = time.time() - _t0
print(f"Computed trend/delta for {test_trend.shape[0]:,} test customers "
      f"({(test_trend.shape[1] - 1)} new columns) in {_test_trend_seconds:.1f}s")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: RATIO FEATURES -- DERIVED FROM NOTEBOOK 02'S EXISTING STATS
# =============================================================================
_section("SECTION 5: Ratio Features")

# --- Cheap, vectorized, no raw re-scan needed -- built directly from the
#     mean/std/min/max/last columns Notebook 02 already computed. Every
#     ratio guards against divide-by-zero and null with pl.when, rather than
#     letting Polars silently produce inf/NaN. ---


def build_ratio_features(df: "pl.DataFrame", numeric_cols: list) -> "pl.DataFrame":
    exprs = []
    new_col_names = []
    for c in numeric_cols:
        _mean, _std, _min, _max, _last = f"{c}_mean", f"{c}_std", f"{c}_min", f"{c}_max", f"{c}_last"
        if not all(col in df.columns for col in (_mean, _std, _min, _max, _last)):
            continue
        _ratio_name, _range_name, _cov_name = (
            f"{c}_ratio_last_to_mean", f"{c}_range", f"{c}_coeff_of_variation",
        )
        exprs.append(
            pl.when((pl.col(_mean).is_not_null()) & (pl.col(_mean) != 0))
            .then(pl.col(_last) / pl.col(_mean))
            .otherwise(None)
            .alias(_ratio_name)
        )
        exprs.append((pl.col(_max) - pl.col(_min)).alias(_range_name))
        exprs.append(
            pl.when((pl.col(_mean).is_not_null()) & (pl.col(_mean) != 0))
            .then(pl.col(_std) / pl.col(_mean))
            .otherwise(None)
            .alias(_cov_name)
        )
        new_col_names += [_ratio_name, _range_name, _cov_name]
    # Apply the expressions against the FULL frame (they reference _mean/_std/
    # _min/_max/_last columns), then select only customer_ID + the new ratio
    # columns -- narrowing to customer_ID first (as an earlier version of this
    # function did) drops those source columns before the expressions can see
    # them, which raises ColumnNotFoundError.
    return df.with_columns(exprs).select(["customer_ID"] + new_col_names)


_t0 = time.time()
train_ratios = build_ratio_features(train_base, numeric_raw_cols)
test_ratios = build_ratio_features(test_base, numeric_raw_cols)
print(f"Built {train_ratios.shape[1] - 1} ratio features (last/mean, range, coefficient of variation) "
      f"for {len(numeric_raw_cols)} numeric columns in {time.time() - _t0:.1f}s")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: INTERACTION TERMS -- GROUNDED IN NOTEBOOK 03'S LIVE CORRELATION RESULTS
# =============================================================================
_section("SECTION 6: Interaction Terms")

# --- Selection is NOT arbitrary: these are the exact columns Notebook 03
#     found, live, to have the highest |correlation with target| (saved in
#     data_validation_report.json). Pairwise interactions among all ~180
#     numeric columns would add ~16,000 columns for no principled reason --
#     restricting to this data-driven top set keeps the expansion small and
#     justified. ---
_top_corr_cols = list(VALIDATION_REPORT["top_5_correlated_with_target"].keys())
_top_corr_cols = [c for c in _top_corr_cols if c in train_base.columns]
print(f"Interaction base columns (top |corr(feature, target)| from Notebook 03): {_top_corr_cols}")

_interaction_exprs = []
_interaction_pairs = []
for i in range(len(_top_corr_cols)):
    for j in range(i + 1, len(_top_corr_cols)):
        c1, c2 = _top_corr_cols[i], _top_corr_cols[j]
        _name = f"interaction_{c1}_x_{c2}"
        _interaction_exprs.append((pl.col(c1) * pl.col(c2)).alias(_name))
        _interaction_pairs.append(_name)

train_interactions = train_base.select(["customer_ID"] + _top_corr_cols).with_columns(_interaction_exprs).select(
    ["customer_ID"] + _interaction_pairs
)
test_interactions = test_base.select(["customer_ID"] + _top_corr_cols).with_columns(_interaction_exprs).select(
    ["customer_ID"] + _interaction_pairs
)
print(f"Built {len(_interaction_pairs)} pairwise interaction terms from {len(_top_corr_cols)} base columns.")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: COMBINE ALL ENGINEERED FEATURES
# =============================================================================
_section("SECTION 7: Combine All Engineered Features")

_t0 = time.time()
train_full_engineered = (
    train_base
    .join(train_trend, on="customer_ID", how="left")
    .join(train_ratios, on="customer_ID", how="left")
    .join(train_interactions, on="customer_ID", how="left")
)
test_engineered = (
    test_base
    .join(test_trend, on="customer_ID", how="left")
    .join(test_ratios, on="customer_ID", how="left")
    .join(test_interactions, on="customer_ID", how="left")
)

if train_full_engineered.shape[0] != train_base.shape[0]:
    raise RuntimeError(
        f"Row count changed after joining engineered features onto train: "
        f"{train_base.shape[0]:,} -> {train_full_engineered.shape[0]:,}. Fix: investigate a "
        f"many-to-one join or customer_ID mismatch before proceeding."
    )
if test_engineered.shape[0] != test_base.shape[0]:
    raise RuntimeError(
        f"Row count changed after joining engineered features onto test: "
        f"{test_base.shape[0]:,} -> {test_engineered.shape[0]:,}."
    )

print(f"train_full_engineered: {train_full_engineered.shape[0]:,} customers x {train_full_engineered.shape[1]} columns "
      f"(was {train_base.shape[1]} before Notebook 04)")
print(f"test_engineered      : {test_engineered.shape[0]:,} customers x {test_engineered.shape[1]} columns "
      f"(was {test_base.shape[1]} before Notebook 04)")
print(f"Combined in {time.time() - _t0:.1f}s")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: DERIVE train_split / test_split ENGINEERED (SAME SPLIT AS NOTEBOOK 02)
# =============================================================================
_section("SECTION 8: Derive Engineered Train/Validation Split")

# --- Reuses the EXACT split membership Notebook 02 already established
#     (read from its output files' customer_ID column) -- this notebook does
#     not re-run train_test_split, so the split stays byte-for-byte
#     consistent between notebooks. ---
_train_split_ids = pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list()
_test_split_ids = pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list()

train_split_engineered = train_full_engineered.filter(pl.col("customer_ID").is_in(_train_split_ids))
test_split_engineered = train_full_engineered.filter(pl.col("customer_ID").is_in(_test_split_ids))

if train_split_engineered.shape[0] != len(_train_split_ids):
    raise RuntimeError(
        f"train_split_engineered row count ({train_split_engineered.shape[0]:,}) does not match "
        f"Notebook 02's train_split.csv customer count ({len(_train_split_ids):,})."
    )
if test_split_engineered.shape[0] != len(_test_split_ids):
    raise RuntimeError(
        f"test_split_engineered row count ({test_split_engineered.shape[0]:,}) does not match "
        f"Notebook 02's test_split.csv customer count ({len(_test_split_ids):,})."
    )

print(f"train_split_engineered: {train_split_engineered.shape[0]:,} customers (matches Notebook 02's split exactly)")
print(f"test_split_engineered : {test_split_engineered.shape[0]:,} customers (matches Notebook 02's split exactly)")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: FEATURE RANKING (REPORTING ONLY -- NO COLUMNS DROPPED HERE)
# =============================================================================
_section("SECTION 9: Feature Ranking by Correlation With Target")

_new_feature_cols = (
    [f"{c}_trend_slope" for c in numeric_raw_cols] + [f"{c}_trend_delta" for c in numeric_raw_cols]
    + [c for c in train_ratios.columns if c != "customer_ID"]
    + _interaction_pairs
)
_new_feature_cols = [c for c in _new_feature_cols if c in train_full_engineered.columns]

_t0 = time.time()
_corr_new = train_full_engineered.select([
    pl.corr(pl.col(c), pl.col("target")).alias(c) for c in _new_feature_cols
]).to_dicts()[0]
_corr_new = {c: v for c, v in _corr_new.items() if v is not None}
_ranked = sorted(_corr_new.items(), key=lambda kv: abs(kv[1]), reverse=True)

feature_ranking_path = PILLAR_DIRS["feature_engineering"] / "engineered_feature_ranking.csv"
pl.DataFrame({"feature": [r[0] for r in _ranked], "corr_with_target": [round(r[1], 5) for r in _ranked]}
             ).write_csv(feature_ranking_path)

print(f"Ranked {len(_ranked)} new engineered features by |corr with target| in {time.time() - _t0:.1f}s")
print("Top 5 new engineered features:")
for name, corr_val in _ranked[:5]:
    print(f"  {name:<40} corr={corr_val:+.4f}")
print(f"\nNote: this is a RANKING, not a deletion -- no column is dropped here. Notebook 05 "
      f"(Model Development) makes the final feature-selection call using model-based importance.")
print(f"\u2705 Saved -> {feature_ranking_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: WRITE ALL OUTPUTS (FIXED PATHS -- OVERWRITE IN PLACE)
# =============================================================================
_section("SECTION 10: Write All Outputs")

FEATURE_ENG_DIR = PILLAR_DIRS["feature_engineering"]

train_full_eng_path = FEATURE_ENG_DIR / "train_full_engineered.parquet"
test_eng_path = FEATURE_ENG_DIR / "test_engineered.parquet"
train_split_eng_path = FEATURE_ENG_DIR / "train_split_engineered.csv"
test_split_eng_path = FEATURE_ENG_DIR / "test_split_engineered.csv"

train_full_engineered.write_parquet(train_full_eng_path)
logger.info(f"Wrote {train_full_eng_path} ({train_full_eng_path.stat().st_size / 1e9:.2f} GB)")

test_engineered.write_parquet(test_eng_path)
logger.info(f"Wrote {test_eng_path} ({test_eng_path.stat().st_size / 1e9:.2f} GB)")

train_split_engineered.write_csv(train_split_eng_path)
logger.info(f"Wrote {train_split_eng_path} ({train_split_eng_path.stat().st_size / 1e9:.2f} GB)")

test_split_engineered.write_csv(test_split_eng_path)
logger.info(f"Wrote {test_split_eng_path} ({test_split_eng_path.stat().st_size / 1e9:.2f} GB)")

print(f"\u2705 {train_full_eng_path.name:<32} -- {train_full_engineered.shape[0]:,} customers x {train_full_engineered.shape[1]} columns")
print(f"\u2705 {test_eng_path.name:<32} -- {test_engineered.shape[0]:,} customers x {test_engineered.shape[1]} columns")
print(f"\u2705 {train_split_eng_path.name:<32} -- {train_split_engineered.shape[0]:,} customers, csv")
print(f"\u2705 {test_split_eng_path.name:<32} -- {test_split_engineered.shape[0]:,} customers, csv")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 10B: VERIFY OUTPUTS
# =============================================================================
_section("SECTION 10B: Verify Outputs Were Written Correctly")

_expected_files = [train_full_eng_path, test_eng_path, train_split_eng_path, test_split_eng_path, feature_ranking_path]
all_ok = True
for fp in _expected_files:
    if fp.exists() and fp.stat().st_size > 0:
        print(f"\u2705 {fp.name:<32} {fp.stat().st_size / 1e6:>10,.1f} MB")
    else:
        all_ok = False
        print(f"\u274c MISSING OR EMPTY: {fp}")

if not all_ok:
    raise RuntimeError("One or more Notebook 04 output files failed to write. See \u274c lines above.")

print("\nAll Notebook 04 outputs verified present and non-empty.")
print("\n\u2705 Section 10B complete.")


# =============================================================================
# SECTION 11: WRITE NOTEBOOK 04 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 11: Write Notebook 04 Summary Artifact")

notebook_04_summary = {
    "notebook": "04_feature_engineering",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "trend_features": {
        "train_seconds": round(_train_trend_seconds, 1),
        "test_seconds": round(_test_trend_seconds, 1),
        "numeric_columns_covered": len(numeric_raw_cols),
    },
    "ratio_features_count": train_ratios.shape[1] - 1,
    "interaction_features_count": len(_interaction_pairs),
    "interaction_base_columns": _top_corr_cols,
    "final_shape": {
        "train_full_engineered": list(train_full_engineered.shape),
        "test_engineered": list(test_engineered.shape),
        "train_split_engineered": list(train_split_engineered.shape),
        "test_split_engineered": list(test_split_engineered.shape),
    },
    "top_5_new_features_by_corr": {name: round(val, 4) for name, val in _ranked[:5]},
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb04_summary_path = ARTIFACTS_DIR / "notebook_04_summary.json"
with open(nb04_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_04_summary, f, indent=2)
print(f"\u2705 Saved -> {nb04_summary_path} (Notebook 17 reads this file to build the rolled-up Feature Engineering section)")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 12: Notebook 04 Complete -- Handoff to Notebook 05")

print("NOTEBOOK 04: FEATURE ENGINEERING -- COMPLETE")
print(f"  Base columns (from Notebook 02)  : {train_base.shape[1]}")
print(f"  + Trend/delta features           : {2 * len(numeric_raw_cols)}")
print(f"  + Ratio features                 : {train_ratios.shape[1] - 1}")
print(f"  + Interaction terms              : {len(_interaction_pairs)}")
print(f"  = Final engineered columns       : {train_full_engineered.shape[1]}")
print(f"  Files produced                   : 5")
for _p in _expected_files:
    print(f"    - {_p.name}")
print(f"  Next notebook                    : 05_model_development.ipynb (Sprint 2)")
print("\n\u2705 Ready to proceed.")
